# 🧬 S1-07: GT Edge & 孤立ノード (Isolated Nodes) 特徴量抽出 & 定量解析 (`s1_07_gt_edge_analysis.ipynb`)

本ノートブックは、Biohub - Cell Tracking During Development コンペティションにおいて、GT (Ground Truth) 時空間グラフのエッジ (Edge) および孤立ノード (Isolated Nodes: 入出力次数0) を対象とし、Zarr画像データから輝度・SNR・Z相対深度・局所密度、および異方性を考慮した物理移動距離 (\mu m) を一括抽出して定量解析を行う Kaggle 専用ノートブックです。

---

## 📦 依存する Kaggle Input Datasets

1. **`biohub-cell-tracking-during-development`** (コンペ公式画像 & GTデータ)
   - パス: `/kaggle/input/competitions/biohub-cell-tracking-during-development/train`
2. **`zarr-offline-installation-wheels`** (Zarr オフラインインストールホイール)
   - パス: `/kaggle/input/datasets/aaaa1597/zarr-offline-installation-wheels/zarr_wheels`
3. **`tracksdata-wheels`** (Tracksdata オフラインインストールホイール)
   - パス: `/kaggle/input/datasets/aaaa1597/tracksdata-wheels`
4. **`btc-s106-progress`** (9時間制限対策・Resume用 Dataset)
   - パス: `/kaggle/input/datasets/aaaa1597/btc-s106-progress`

---

## 📁 出力成果物 (Output Artifacts)

- `/kaggle/working/gt_edges_summary.xlsx` (Excel形式: GT_Edges_Summary, Isolated_Nodes, Node_Summary の複数シート統合)
- `/kaggle/working/gt_edges_summary.csv` (エッジ主軸 CSV)
- `/kaggle/working/gt_edges_summary.parquet` (エッジ主軸 Parquet)
- `/kaggle/working/gt_isolated_nodes_summary.xlsx` (孤立ノード Excel)
- `/kaggle/working/gt_isolated_nodes_summary.csv` (孤立ノード CSV)
- `/kaggle/working/gt_isolated_nodes_summary.parquet` (孤立ノード Parquet)
- `/kaggle/working/progress.json` (途中保存・Resume管理JSON)
- `/kaggle/working/gt_edges_eda_plot.png` (★3×3 拡張マルチパネル EDA 可視化プロット)
- `/kaggle/working/gt_isolated_nodes_eda_plot.png` (孤立ノード vs 接続ノード 比較 EDA プロット)

## 📁 コーディングルール
- 基本例外はキャッチしない。その例外が発生しても無視していい時のみキャッチする。
- パス探索はしない。ライブラリが見つからないときは環境構築に失敗している。
- 環境構築やパッケージ配置に不足があれば、即座に ModuleNotFoundError、ImportError をスローする。


## 🗺️ 処理フローチャート (Pipeline Flowchart)

```mermaid
graph TD
    classDef default fill:#f9f9f9,stroke:#333,stroke-width:1px;
    classDef loop fill:#e1f5fe,stroke:#0288d1,stroke-width:2px;
    classDef func fill:#efebe9,stroke:#5d4037,stroke-width:1px;
    classDef cond fill:#fff9c4,stroke:#fbc02d,stroke-width:1px;

    Start([処理開始]) --> Cell3["Cell 3: パラメータ設定 & check_environment()"]
    Cell3 --> Cell4["Cell 4: オフラインライブラリの自動インストール"]
    Cell4 --> Cell5["Cell 5: GTEdgeExtractor コア関数の定義"]
    Cell5 --> Cell6_Init["Cell 6: 初期化 & Resume復元 (progress.json)"]

    subgraph Loop ["Cell 6: データセットごとの一括処理ループ"]
        LoopStart{"データセット処理開始"}
        class LoopStart loop;
        
        LoopStart --> CheckSkip{"CONTINUOUS_FLAG == True <br>&& 処理済みデータセット?"}
        class CheckSkip cond;
        
        CheckSkip -- Yes (スキップ) --> LoopNext
        CheckSkip -- No --> LoadZarr["1. Zarr画像 & .geff Tracks ロード"]
        LoadZarr --> ExtractEdges["2. GTエッジ空間座標 & 異方性物理距離算出"]
        ExtractEdges --> CalcSignal["3. 3Dボクセル参照 & 背景球殻SNR抽出"]
        CalcSignal --> SaveProgress["4. 途中経過保存 (progress.json)"]
        SaveProgress --> LoopNext
    end

    Cell6_Init --> LoopStart
    LoopNext{"次のデータセットあり?"}
    class LoopNext loop;
    LoopNext -- Yes --> LoopStart
    
    LoopNext -- No --> Cell7["Cell 7: データ結合 & プロ仕様Excel/CSV/Parquet保存"]
    Cell7 --> Cell8["Cell 8: 簡易EDA・可視化 & サマリ表示"]
    Cell8 --> End([処理終了])
```


In [ ]:
# === Cell 3: パラメータ設定 & 環境確認関数の定義 (check_environment) ===
import datetime
import os
import sys
import glob

print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] >>> Cell 3: パラメータ設定 開始")

# ==========================================
#   CONFIGURATION PARAMETERS(設定パラメータ)
# ==========================================
RESET_CHECKPOINT = False  # True: 過去の進捗をクリアして新規スタート
MAX_FRAMES = None          # デバッグ時(例: 5) または 全フレーム処理(None)
TARGET_DATASETS = []       # 特定データセットのみ指定する場合(例: ['6bba_0c7fa718']), 空リストで全件

# GitHub 自動プッシュ & リポジトリ設定
PUSH_TO_GITHUB = False     # GitHub 自動プッシュ有効化フラグ
GITHUB_REPO = "aaaa1597/kaggle_Biohub-Cell_Tracking_During_Development"

# 1. 物理スケール定数 (Z, Y, X μm/voxel)
SCALE_Z = 1.625
SCALE_Y = 0.40625
SCALE_X = 0.40625

# 2. SNR・輝度抽出パラメータ (ボクセル半径)
NODE_R = 4.0        # 内側細胞領域球体半径 (px)
BG_R_IN = 8.0       # 近傍背景球殻内径 (px)
BG_R_OUT = 12.0     # 近傍背景球殻外径 (px)
DENSITY_RADIUS = 15.0 # 局所密度カウント半径 (px)

# 3. 途中保存 & Resume 設定 (9時間制限対策)
CONTINUOUS_FLAG = True
DATASET_SLUG = "btc-s106-progress"
DATA_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"
CHECKPOINT_DATASET_PATH = f"/kaggle/input/datasets/aaaa1597/{DATASET_SLUG}"

def check_environment():
    """
    Kaggle実行環境の整合性チェックおよび入力データディレクトリの存在確認を行う。
    実行条件が満たされていない場合は例外(FileNotFoundError/RuntimeError)を送出、中止する。
    
    Args:
        none.
        
    Returns:
        none.
        
    Raises:
        FileNotFoundError: 入力データディレクトリが存在しない場合
        RuntimeError: ディレクトリ内に対象データが存在しない場合
    """
    print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] >>> check_environment 開始")
    print(f"Pythonバージョン: {sys.version}")
    print(f"カレント作業ディレクトリ: {os.getcwd()}")
    
    # 1. 共通コアライブラリのインポートチェック
    try:
        import zarr
        import openpyxl
        import pandas as pd
        import numpy as np
        print("  - [OK] Core libraries (zarr, openpyxl, pandas, numpy) are installed.")
    except ImportError as e:
        raise ImportError(f"❌ [ERROR] 必須コアライブラリ '{e.name}' のインポートに失敗しました。") from e

    # 2. データディレクトリの存在チェック
    if DATA_DIR is None or not os.path.exists(DATA_DIR):
        error_msg = f"❌ [ERROR] 必須の入力データディレクトリが存在しません: {DATA_DIR}\nKaggle Dataset または コンペデータが正しく追加されているか確認してください。"
        print(error_msg)
        raise FileNotFoundError(error_msg)
    datasets = [d for d in os.listdir(DATA_DIR) if d.endswith('.zarr') or d.endswith('.geff')]
    if len(datasets) == 0:
        error_msg = f"❌ [ERROR] 入力ディレクトリ ({DATA_DIR}) 内に対象データファイル (.zarr / .geff) が見つかりません。"
        print(error_msg)
        raise RuntimeError(error_msg)
    print(f"  - [OK] 入力データディレクトリ確認成功: {DATA_DIR} (発見ファイル数: {len(datasets)})")

    # 3. GitHub 自動プッシュ設定のチェック (PUSH_TO_GITHUB=True の時のみ必須)
    if PUSH_TO_GITHUB:
        if not GITHUB_REPO or "/" not in GITHUB_REPO:
            raise ValueError(f"❌ [ERROR] PUSH_TO_GITHUB is True, but GITHUB_REPO '{GITHUB_REPO}' is invalid. Expected format 'username/repository'.")
        
        github_token = None
        is_kaggle = os.path.exists("/kaggle/working")
        if is_kaggle:
            try:
                from kaggle_secrets import UserSecretsClient
                github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
            except Exception as e_sec:
                raise ValueError(f"❌ [ERROR] PUSH_TO_GITHUB is True on Kaggle, but failed to fetch GITHUB_TOKEN from Kaggle Secrets: {e_sec}") from e_sec
        else:
            github_token = os.getenv("GITHUB_TOKEN")

        if not github_token or len(str(github_token).strip()) == 0:
            raise ValueError("❌ [ERROR] PUSH_TO_GITHUB is True, but GITHUB_TOKEN is not found or empty in Kaggle Secrets or environment variables.")
        print(f"  - [OK] GitHub Auto-Push configuration (repo: '{GITHUB_REPO}', token: verified) is valid.")

    # 4. Kaggle Dataset 自動書き込み同期設定のチェック (CONTINUOUS_FLAG=True かつ Kaggle本番時)
    if CONTINUOUS_FLAG:
        if not DATASET_SLUG:
            raise ValueError("❌ [ERROR] CONTINUOUS_FLAG is True, but DATASET_SLUG is not defined.")
        
        is_kaggle = os.path.exists("/kaggle/working")
        if is_kaggle:
            try:
                from kaggle_secrets import UserSecretsClient
                u = UserSecretsClient().get_secret("KAGGLE_USERNAME")
                k = UserSecretsClient().get_secret("KAGGLE_KEY")
                if not u or not k:
                    raise ValueError("Empty credentials in Kaggle Secrets.")
            except Exception as e_k:
                raise ValueError(f"❌ [ERROR] CONTINUOUS_FLAG is True on Kaggle, but KAGGLE_USERNAME or KAGGLE_KEY is missing/invalid in Secrets: {e_k}") from e_k
            print(f"  - [OK] Kaggle Dataset Checkpoint Auto-Sync (slug: '{DATASET_SLUG}', credentials: verified) is valid.")

    print("✅ Environment check passed successfully!")
    print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] <<< check_environment 終了")

print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] <<< Cell 3: パラメータ設定 終了")


In [ ]:
# === Cell 4: オフラインライブラリ自動インストール & 環境チェック ===
import datetime
import os
import sys
import subprocess

print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] >>> Cell 4: ライブラリ自動インストール 開始")

# === Polars の Float16 欠損に対するモンキーパッチ (tracksdata インポート前に必須) ===
import polars as pl
if not hasattr(pl, 'Float16'):
    pl.Float16 = pl.Float32

# 1. zarr オフラインインストール
ZARR_WHEELS_PATH = "/kaggle/input/datasets/aaaa1597/zarr-offline-installation-wheels/zarr_wheels"
if os.path.exists(ZARR_WHEELS_PATH):
    print(f"zarr オフラインホイールをインストール中 ({ZARR_WHEELS_PATH})...")
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", f"--find-links={ZARR_WHEELS_PATH}", "zarr"], check=True)
    print("✅ zarr インストール完了")
else:
    raise FileNotFoundError(f"❌ [ERROR] Zarr ホイールパスが存在しません: {ZARR_WHEELS_PATH}")

# 2. tracksdata オフラインインストール
TRACKSDATA_WHEELS_PATH = "/kaggle/input/datasets/aaaa1597/tracksdata-wheels"
if os.path.exists(TRACKSDATA_WHEELS_PATH):
    print(f"tracksdata オフラインホイールをインストール中 ({TRACKSDATA_WHEELS_PATH})...")
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", f"--find-links={TRACKSDATA_WHEELS_PATH}", "rustworkx", "bidict", "ilpy", "imagecodecs", "polars", "btrack", "zarr"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", f"--find-links={TRACKSDATA_WHEELS_PATH}", "geff", "geff-spec"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", f"--find-links={TRACKSDATA_WHEELS_PATH}", "tracksdata", "openpyxl"], check=True)
    print("✅ tracksdata インストール完了")
else:
    raise FileNotFoundError(f"❌ [ERROR] tracksdata ホイールパスが存在しません: {TRACKSDATA_WHEELS_PATH}")

# ライブラリ自動インストール完了後に環境チェックを実行
check_environment()

print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] <<< Cell 4: ライブラリ自動インストール 終了")


In [ ]:
# === Cell 5: GT Edge & ノード特徴量抽出コア関数群の定義 ===
import datetime
import numpy as np
import pandas as pd
import polars as pl
import os
import sys

print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] >>> Cell 5: コア関数群定義 開始")

class GTEdgeExtractor:
    """
    GT (Ground Truth) Tracks Graph および Zarr画像から、
    エッジ主軸および孤立ノードの時空間特徴量（輝度、SNR、Z相対深度、局所密度、異方性物理距離、動的細胞体積・半径）を
    高速にバッチ抽出する計算クラス。
    """
    def __init__(self, scale_z=1.625, scale_y=0.40625, scale_x=0.40625):
        """
        初期化パラメータ設定。
        
        Args:
            scale_z (float): Z軸物理スケール (μm/voxel)
            scale_y (float): Y軸物理スケール (μm/voxel)
            scale_x (float): X軸物理スケール (μm/voxel)
        """
        self.scale_z = scale_z
        self.scale_y = scale_y
        self.scale_x = scale_x
        self.voxel_volume_um3 = scale_z * scale_y * scale_x  # 1ボクセルの物理体積 (μm^3)

    def compute_anisotropic_distance_um(self, s_z, s_y, s_x, e_z, e_y, e_x):
        """
        異方性ボクセル分解能を考慮した 3D 直線物理移動距離 (μm) を算出する。
        
        Args:
            s_z, s_y, s_x (float/np.ndarray): 始点ノード座標
            e_z, e_y, e_x (float/np.ndarray): 終点ノード座標
            
        Returns:
            np.ndarray: 物理移動距離 (μm)
        """
        dz = (e_z - s_z) * self.scale_z
        dy = (e_y - s_y) * self.scale_y
        dx = (e_x - s_x) * self.scale_x
        return np.sqrt(dz**2 + dy**2 + dx**2)

    def extract_node_signal_features(self, img_3d, coords, r_node=4.0, bg_r_in=8.0, bg_r_out=12.0):
        """
        3D画像ボリュームとノード座標配列から、細胞内部の平均輝度・SNR・Z相対深度・細胞ごとの動的物理体積(μm^3)・物理半径(μm)を一元算出する。
        
        Args:
            img_3d (np.ndarray): 3D画像配列 (Z, Y, X)
            coords (np.ndarray): ノードの 3D 座標 (N, 3) -> [z, y, x]
            r_node (float): 細胞内部領域球体半径
            bg_r_in (float): 近傍背景球殻内径
            bg_r_out (float): 近傍背景球殻外径
            
        Returns:
            tuple: (mean_intensities, snrs, z_depth_ratios) 各 (N,) の配列
        """
        Z, Y, X = img_3d.shape
        n_nodes = len(coords)
        
        mean_intensities = np.zeros(n_nodes, dtype=np.float32)
        snrs = np.zeros(n_nodes, dtype=np.float32)
        z_depth_ratios = np.zeros(n_nodes, dtype=np.float32)
        volumes_um3 = np.zeros(n_nodes, dtype=np.float32)
        radii_um = np.zeros(n_nodes, dtype=np.float32)
        
        if n_nodes == 0:
            return mean_intensities, snrs, z_depth_ratios, volumes_um3, radii_um
            
        for i, (z, y, x) in enumerate(coords):
            z_depth_ratios[i] = float(z) / max(1.0, float(Z - 1))
            
            iz, iy, ix = int(round(z)), int(round(y)), int(round(x))
            
            z_min, z_max = max(0, int(iz - bg_r_out)), min(Z, int(iz + bg_r_out + 1))
            y_min, y_max = max(0, int(iy - bg_r_out)), min(Y, int(iy + bg_r_out + 1))
            x_min, x_max = max(0, int(ix - bg_r_out)), min(X, int(ix + bg_r_out + 1))
            
            patch = img_3d[z_min:z_max, y_min:y_max, x_min:x_max]
            if patch.size == 0:
                continue
                
            zz, yy, xx = np.ogrid[z_min-iz:z_max-iz, y_min-iy:y_max-iy, x_min-ix:x_max-ix]
            dist_sq = zz**2 + yy**2 + xx**2
            
            # ① 細胞内部マスク
            node_mask = dist_sq <= (r_node**2)
            # ② 背景球殻マスク
            bg_mask = (dist_sq >= (bg_r_in**2)) & (dist_sq <= (bg_r_out**2))
            
            cell_vals = patch[node_mask]
            bg_vals = patch[bg_mask]
            
            mean_intensity = np.mean(cell_vals) if len(cell_vals) > 0 else float(img_3d[iz, iy, ix])
            bg_mean = np.mean(bg_vals) if len(bg_vals) > 0 else 0.0
            bg_std = np.std(bg_vals) if len(bg_vals) > 0 else 1.0
            
            snr = (mean_intensity - bg_mean) / (bg_std + 1e-5)
            
            # 🌟 細胞ごとの動的領域ボクセル数の算出 (背景を超える輝度ボクセル数)
            cell_threshold = bg_mean + 0.2 * max(1e-5, mean_intensity - bg_mean)
            cell_voxels = np.sum((patch >= cell_threshold) & node_mask)
            if cell_voxels == 0:
                cell_voxels = max(1, len(cell_vals))
                
            vol = cell_voxels * self.voxel_volume_um3
            rad = ((3.0 * vol) / (4.0 * np.pi)) ** (1.0 / 3.0)
            
            mean_intensities[i] = mean_intensity
            snrs[i] = snr
            volumes_um3[i] = vol
            radii_um[i] = rad
            
        return mean_intensities, snrs, z_depth_ratios, volumes_um3, radii_um

    def compute_local_density(self, coords, radius=15.0):
        """
        同一フレーム内の指定ノード周辺（半径15px内）のGT細胞密度を求める。
        
        Args:
            coords (np.ndarray): フレーム内の全GT細胞3D座標 (N, 3)
            radius (float): 密度測定半径
            
        Returns:
            np.ndarray: 各ノードの近傍細胞数 (N,)
        """
        N = len(coords)
        densities = np.zeros(N, dtype=np.int32)
        if N <= 1:
            return densities
            
        diff = coords[:, None, :] - coords[None, :, :]
        dist = np.sqrt(np.sum(diff**2, axis=-1))
        densities = np.sum((dist <= radius) & (dist > 0), axis=1)
        return densities

print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] <<< Cell 5: コア関数群定義 終了")


In [ ]:
# === Cell 6: 一括抽出処理ループ & Resume (途中保存) 復元機能 ===
import datetime
import os
import sys
import json
import numpy as np
import pandas as pd
import torch

print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] >>> Cell 6: 一括抽出ループ 開始")

# Kaggle Input Dataset の固定ソースパスを sys.path に追加
KAGGLE_SRC = "/kaggle/input/datasets/aaaa1597/kaggle-cell-tracking-competition/src"
if os.path.exists(KAGGLE_SRC) and KAGGLE_SRC not in sys.path:
    sys.path.insert(0, KAGGLE_SRC)

import tracking_cellmot.io

# === GPUなし環境での RuntimeError 対策 (CPUモンキーパッチ) ===
if not torch.cuda.is_available():
    print("GPU is not available. Applying CPU monkey patch to tracking_cellmot.io._process_on_gpu...")
    
    def patched_process_on_gpu(
        image, tracks, scale, device,
        resample=False, target_scale=None,
        normalize=True, gamma=1.0,
        q_min=0.01, q_max=0.99, subsample_factor=1000,
        precomputed_quantiles=None
    ):
        torch_device = torch.device(device)
        image = image.astype(np.float32, copy=False)
        q1, q2 = None, None
        if normalize:
            q1 = tracking_cellmot.io._lookup_precomputed_quantile(precomputed_quantiles, q_min)
            q2 = tracking_cellmot.io._lookup_precomputed_quantile(precomputed_quantiles, q_max)
            if q1 is None or q2 is None:
                flat = image.ravel()[::subsample_factor]
                q1, q2 = np.quantile(flat, [q_min, q_max]).astype(np.float32)
            else:
                q1 = np.float32(q1)
                q2 = np.float32(q2)
        tensor = torch.from_numpy(image)
        tensor = tensor.to(torch_device, non_blocking=True)
        if normalize:
            tensor = (tensor - float(q1)) / (float(q2) - float(q1) + 1e-6)
            tensor = tensor.clamp(min=0.0)
            if gamma != 1.0:
                tensor = tensor.pow(gamma)
            tensor = tensor.clamp(0.0, 4.0)
        if resample:
            scale_arr = np.array(scale)
            target_scale_val = scale_arr.min() if target_scale is None else np.array(target_scale)
            zoom_factors = scale_arr / target_scale_val
            new_spatial_shape = (np.array(tensor.shape[1:]) * zoom_factors).astype(int).tolist()
            tensor = tensor[:, None]
            tensor = torch.nn.functional.interpolate(
                tensor, size=new_spatial_shape, mode="trilinear", align_corners=False
            )
            tensor = tensor[:, 0]
            if tracks is not None:
                import tracksdata as td
                import polars as pl
                node_attrs = tracks.node_attrs()
                orig_dtypes = {col: node_attrs.schema[col] for col in ["z", "y", "x"]}
                node_attrs = node_attrs.with_columns(
                    (pl.col("z") * zoom_factors[0]).round(0).cast(orig_dtypes["z"]),
                    (pl.col("y") * zoom_factors[1]).round(0).cast(orig_dtypes["y"]),
                    (pl.col("x") * zoom_factors[2]).round(0).cast(orig_dtypes["x"]),
                )
                tracks.update_node_attrs(
                    attrs=node_attrs.select("z", "y", "x").to_dict(),
                    node_ids=node_attrs[td.DEFAULT_ATTR_KEYS.NODE_ID].to_list(),
                )
            scale = target_scale if target_scale is not None else (float(target_scale_val),) * 3
        return tensor, tracks, scale
    
    tracking_cellmot.io._process_on_gpu = patched_process_on_gpu
    print("✅ CPU Monkey Patch applied to tracking_cellmot.io._process_on_gpu!")

from tracking_cellmot.io import open_dataset

# 1. 進捗管理プログレスファイルの初期化 / 復元
progress_file = "progress.json"
processed_datasets = []

if RESET_CHECKPOINT and os.path.exists(progress_file):
    os.remove(progress_file)
    print("チェックポイントをクリアしました。")

if os.path.exists(progress_file):
    with open(progress_file, "r") as f:
        progress_data = json.load(f)
        processed_datasets = progress_data.get("processed_datasets", [])
        print(f"✅ 過去の進捗を復元: 処理済みデータセット数 {len(processed_datasets)}")
elif os.path.exists(os.path.join(CHECKPOINT_DATASET_PATH, progress_file)):
    with open(os.path.join(CHECKPOINT_DATASET_PATH, progress_file), "r") as f:
        progress_data = json.load(f)
        processed_datasets = progress_data.get("processed_datasets", [])
        print(f"✅ Input Dataset から進捗を復元: 処理済み {len(processed_datasets)} 件")

data_dir = DATA_DIR
dataset_names = []
if os.path.exists(data_dir):
    files = os.listdir(data_dir)
    dataset_names = sorted(list(set([f.split('.')[0] for f in files if f.endswith('.zarr') or f.endswith('.geff')])))

if TARGET_DATASETS:
    dataset_names = [d for d in dataset_names if d in TARGET_DATASETS]

total_ds = len(dataset_names)
print(f"処理対象データセット一覧 ({total_ds} 件)")

extractor = GTEdgeExtractor(scale_z=SCALE_Z, scale_y=SCALE_Y, scale_x=SCALE_X)
all_edge_records = []
all_isolated_node_records = []
node_summary_records = []

# === 固定の決定済みファイルパスからの復元 (パス探索ループを全廃) ===
summary_csv_file = "gt_edges_summary.csv"
summary_parquet_file = "gt_edges_summary.parquet"
iso_csv_file = "gt_isolated_nodes_summary.csv"
iso_parquet_file = "gt_isolated_nodes_summary.parquet"
loaded_existing = False

if not RESET_CHECKPOINT and CONTINUOUS_FLAG:
    target_edge_path = summary_csv_file if os.path.exists(summary_csv_file) else os.path.join(CHECKPOINT_DATASET_PATH, summary_csv_file)
    target_iso_path = iso_csv_file if os.path.exists(iso_csv_file) else os.path.join(CHECKPOINT_DATASET_PATH, iso_csv_file)
    
    if os.path.exists(target_edge_path):
        df_prev = pd.read_csv(target_edge_path)
        all_edge_records = df_prev.to_dict('records')
        loaded_existing = True
        print(f"✅ 過去の成果物 ({target_edge_path}) から {len(all_edge_records)} 件のエッジレコードを復元しました。")
        
    if os.path.exists(target_iso_path):
        df_iso_prev = pd.read_csv(target_iso_path)
        all_isolated_node_records = df_iso_prev.to_dict('records')
        print(f"✅ 過去の成果物 ({target_iso_path}) から {len(all_isolated_node_records)} 件の孤立ノードレコードを復元しました。")

# レコード内に存在する処理済みデータセット名の集合
records_datasets = set(r['dataset'] for r in all_edge_records) if all_edge_records else set()

# 処理ループ
for i, ds_name in enumerate(dataset_names, 1):
    if CONTINUOUS_FLAG and (ds_name in processed_datasets) and (ds_name in records_datasets or not loaded_existing):
        print(f"⏩ [Skip] ({i}/{total_ds}) データセット {ds_name} は処理済みのためスキップします。")
        continue
        
    print(f"\n--- データセット処理中 ({i}/{total_ds}): {ds_name} ---")
    ds_path = os.path.join(data_dir, ds_name)
    
    ds = open_dataset(ds_path, normalize=True, require_tracks=True, device="cpu")
    tracks_graph = ds.tracks
    img_tensor = getattr(ds, 'image', None)
    if img_tensor is not None and hasattr(img_tensor, 'numpy'):
        img_data = img_tensor.numpy()
    elif img_tensor is not None:
        img_data = np.array(img_tensor)
    else:
        img_data = None
        
    nodes_df = tracks_graph.node_attrs().to_pandas()
    edges_df = tracks_graph.edge_attrs().to_pandas()
    
    # ノード辞書の高速作成 (itertuples)
    node_dict = {row.node_id: row for row in nodes_df.itertuples(index=False)}
    
    connected_node_ids = set(edges_df['source_id']) | set(edges_df['target_id'])
    all_node_ids = set(nodes_df['node_id'])
    isolated_node_ids = all_node_ids - connected_node_ids
    
    num_total_nodes = len(all_node_ids)
    num_connected_nodes = len(connected_node_ids)
    num_isolated_nodes = len(isolated_node_ids)
    iso_ratio = (num_isolated_nodes / num_total_nodes * 100.0) if num_total_nodes > 0 else 0.0
    
    node_summary_records.append({
        'dataset': ds_name,
        'total_nodes': num_total_nodes,
        'connected_nodes': num_connected_nodes,
        'isolated_nodes': num_isolated_nodes,
        'isolated_ratio_pct': iso_ratio,
        'total_edges': len(edges_df)
    })
    
    # 🌟 各 source_id の出次数 (1つの親から出ているエッジの本数) を事前計算して分裂判定に利用
    source_counts = edges_df['source_id'].value_counts().to_dict()
    
    # 🌟 細胞ごとの個別・動的特徴量の計算
    node_features_map = {}
    for t_val, t_group in nodes_df.groupby('t'):
        t_int = int(t_val)
        coords = t_group[['z', 'y', 'x']].values.astype(np.float32)
        
        # 1. 局所密度
        densities = extractor.compute_local_density(coords, radius=DENSITY_RADIUS)
        
        # 2. 輝度・SNR・Z相対深度・個別の動的体積(μm^3)・動的半径(μm)
        if img_data is not None and t_int < len(img_data):
            img_3d = img_data[t_int]
            intensities, snrs, z_depths, vols, rads = extractor.extract_node_signal_features(
                img_3d, coords, r_node=NODE_R, bg_r_in=BG_R_IN, bg_r_out=BG_R_OUT
            )
        else:
            intensities = np.full(len(coords), -1.0, dtype=np.float32)
            snrs = np.full(len(coords), -1.0, dtype=np.float32)
            z_max_val = max(1.0, float(nodes_df['z'].max()))
            z_depths = (coords[:, 0] / z_max_val).astype(np.float32)
            vols = np.full(len(coords), -1.0, dtype=np.float32)
            rads = np.full(len(coords), -1.0, dtype=np.float32)
            
        for idx_n, row_n in enumerate(t_group.itertuples(index=False)):
            node_features_map[row_n.node_id] = {
                'mean_intensity': float(intensities[idx_n]),
                'snr': float(snrs[idx_n]),
                'z_depth_ratio': float(z_depths[idx_n]),
                'estimated_radius_um': float(rads[idx_n]),
                'volume_um3': float(vols[idx_n]),
                'local_density_r15': int(densities[idx_n])
            }
            
    print(f"総ノード数: {num_total_nodes} (接続: {num_connected_nodes}, 孤立: {num_isolated_nodes} [{iso_ratio:.2f}%]), エッジ数: {len(edges_df)}")
    
    # ① エッジ一括処理
    for edge in edges_df.itertuples(index=False):
        s_id = edge.source_id
        e_id = edge.target_id
        
        s_node = node_dict.get(s_id)
        e_node = node_dict.get(e_id)
        
        if s_node is None or e_node is None:
            continue
            
        s_feat = node_features_map.get(s_id, {})
        e_feat = node_features_map.get(e_id, {})
        
        s_z, s_y, s_x = s_node.z, s_node.y, s_node.x
        e_z, e_y, e_x = e_node.z, e_node.y, e_node.x
        
        dist_um = extractor.compute_anisotropic_distance_um(s_z, s_y, s_x, e_z, e_y, e_x)
        
        # 親ノード s_id から出ているエッジが 2本以上なら 'division' (分裂)、1本なら 'move' (移動)
        edge_type = 'division' if source_counts.get(s_id, 0) > 1 else 'move'
        
        all_edge_records.append({
            'edge_id': edge.edge_id,
            'dataset': ds_name,
            'edge_type': edge_type,
            't': int(s_node.t),
            's_node_id': s_id,
            's_z': s_z,
            's_y': s_y,
            's_x': s_x,
            's_mean_intensity': s_feat.get('mean_intensity', -1.0),
            's_snr': s_feat.get('snr', -1.0),
            's_z_depth_ratio': s_feat.get('z_depth_ratio', s_z / 30.0),
            's_estimated_radius_um': s_feat.get('estimated_radius_um', -1.0),
            's_volume_um3': s_feat.get('volume_um3', -1.0),
            's_local_density_r15': s_feat.get('local_density_r15', -1),
            'e_node_id': e_id,
            'e_z': e_z,
            'e_y': e_y,
            'e_x': e_x,
            'e_mean_intensity': e_feat.get('mean_intensity', -1.0),
            'e_snr': e_feat.get('snr', -1.0),
            'e_z_depth_ratio': e_feat.get('z_depth_ratio', e_z / 30.0),
            'e_estimated_radius_um': e_feat.get('estimated_radius_um', -1.0),
            'e_volume_um3': e_feat.get('volume_um3', -1.0),
            'e_local_density_r15': e_feat.get('local_density_r15', -1),
            'track_distance_3d_um': dist_um
        })
        
    # ② 孤立ノード一括処理
    for iso_id in isolated_node_ids:
        iso_node = node_dict.get(iso_id)
        if iso_node is None:
            continue
        iso_feat = node_features_map.get(iso_id, {})
        all_isolated_node_records.append({
            'dataset': ds_name,
            'node_id': iso_id,
            't': int(iso_node.t),
            'z': iso_node.z,
            'y': iso_node.y,
            'x': iso_node.x,
            'mean_intensity': iso_feat.get('mean_intensity', -1.0),
            'snr': iso_feat.get('snr', -1.0),
            'z_depth_ratio': iso_feat.get('z_depth_ratio', iso_node.z / 30.0),
            'estimated_radius_um': iso_feat.get('estimated_radius_um', -1.0),
            'volume_um3': iso_feat.get('volume_um3', -1.0),
            'local_density_r15': iso_feat.get('local_density_r15', -1)
        })
        
    processed_datasets.append(ds_name)
    with open(progress_file, "w") as f:
        json.dump({"processed_datasets": processed_datasets}, f, indent=2)
    records_datasets.add(ds_name)
    
    # 途中保存: 万一途中でセッションが終了しても次回セッションへ引き継げるようにエッジデータを途中書き出し
    if len(all_edge_records) > 0:
        pd.DataFrame(all_edge_records).to_csv(summary_csv_file, index=False)
        pd.DataFrame(all_edge_records).to_parquet(summary_parquet_file, index=False)
    if len(all_isolated_node_records) > 0:
        pd.DataFrame(all_isolated_node_records).to_csv(iso_csv_file, index=False)
        pd.DataFrame(all_isolated_node_records).to_parquet(iso_parquet_file, index=False)
            
    print(f"✅ データセット ({i}/{total_ds}) {ds_name} 完了 & 進捗保存")

print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] <<< Cell 6: 一括抽出ループ 終了")


In [ ]:
# === Cell 7: 最終データ結合 & プロ仕様 Excel / CSV / Parquet 出力 ===
import datetime
import os
import sys
import json
import shutil
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.comments import Comment

print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] >>> Cell 7: 最終データ保存 開始")

if len(all_edge_records) == 0:
    raise RuntimeError('❌ [ERROR] 想定外!! GTエッジデータが空。')

df_edges = pd.DataFrame(all_edge_records)
df_isolated = pd.DataFrame(all_isolated_node_records) if len(all_isolated_node_records) > 0 else pd.DataFrame()
df_node_summary = pd.DataFrame(node_summary_records) if len(node_summary_records) > 0 else pd.DataFrame()

# カラム順序の統一定義 (e_z_depth_ratio も含める)
columns_order_edges = [
    'edge_id', 'dataset', 'edge_type', 't',
    's_node_id', 's_z', 's_y', 's_x', 's_mean_intensity', 's_snr', 's_z_depth_ratio', 's_estimated_radius_um', 's_volume_um3', 's_local_density_r15',
    'e_node_id', 'e_z', 'e_y', 'e_x', 'e_mean_intensity', 'e_snr', 'e_z_depth_ratio', 'e_estimated_radius_um', 'e_volume_um3', 'e_local_density_r15',
    'track_distance_3d_um'
]
columns_order_edges = [c for c in columns_order_edges if c in df_edges.columns]
df_edges = df_edges[columns_order_edges]

# 1. CSV / Parquet 出力
csv_path = "gt_edges_summary.csv"
parquet_path = "gt_edges_summary.parquet"
iso_csv_path = "gt_isolated_nodes_summary.csv"
iso_parquet_path = "gt_isolated_nodes_summary.parquet"
excel_path = "gt_edges_summary.xlsx"
iso_excel_path = "gt_isolated_nodes_summary.xlsx"

df_edges.to_csv(csv_path, index=False)
df_edges.to_parquet(parquet_path, index=False)
print(f"✅ エッジ CSV & Parquet 保存完了: {csv_path} ({len(df_edges)} 行)")

if not df_isolated.empty:
    df_isolated.to_csv(iso_csv_path, index=False)
    df_isolated.to_parquet(iso_parquet_path, index=False)
    print(f"✅ 孤立ノード CSV & Parquet 保存完了: {iso_csv_path} ({len(df_isolated)} 行)")

# ----------------------------------------------------
# 🎨 Excel 出力 & openpyxl スタイリング装飾 & 複数シート作成
# ----------------------------------------------------
wb = openpyxl.Workbook()

fill_common = PatternFill(start_color="D3D3D3", end_color="D3D3D3", fill_type="solid")
fill_start  = PatternFill(start_color="E2EFDA", end_color="E2EFDA", fill_type="solid")
fill_end    = PatternFill(start_color="FCE4D6", end_color="FCE4D6", fill_type="solid")
fill_motion = PatternFill(start_color="FFF2CC", end_color="FFF2CC", fill_type="solid")
fill_iso    = PatternFill(start_color="F2DCDB", end_color="F2DCDB", fill_type="solid")

font_header = Font(name="Segoe UI", size=11, bold=True, color="000000")
align_center = Alignment(horizontal="center", vertical="center")
thin_border = Border(left=Side(style='thin', color='BFBFBF'), right=Side(style='thin', color='BFBFBF'), top=Side(style='thin', color='BFBFBF'), bottom=Side(style='thin', color='BFBFBF'))

# Sheet 1: GT_Edges_Summary
ws_edges = wb.active
ws_edges.title = "GT_Edges_Summary"
ws_edges.append(list(df_edges.columns))
for row in df_edges.itertuples(index=False):
    ws_edges.append(list(row))
for col_idx, col_name in enumerate(df_edges.columns, 1):
    cell = ws_edges.cell(row=1, column=col_idx)
    cell.font = font_header
    cell.alignment = align_center
    cell.border = thin_border
    cell.fill = fill_start if col_name.startswith('s_') else (fill_end if col_name.startswith('e_') else (fill_common if col_name in ['edge_id', 'dataset', 'edge_type', 't'] else fill_motion))
    ws_edges.column_dimensions[get_column_letter(col_idx)].width = 16

# Sheet 2: Isolated_Nodes
if not df_isolated.empty:
    ws_iso = wb.create_sheet(title="Isolated_Nodes")
    ws_iso.append(list(df_isolated.columns))
    for row in df_isolated.itertuples(index=False):
        ws_iso.append(list(row))
    for col_idx, col_name in enumerate(df_isolated.columns, 1):
        cell = ws_iso.cell(row=1, column=col_idx)
        cell.font = font_header
        cell.alignment = align_center
        cell.border = thin_border
        cell.fill = fill_iso
        ws_iso.column_dimensions[get_column_letter(col_idx)].width = 16

# Sheet 3: Node_Summary
if not df_node_summary.empty:
    ws_sum = wb.create_sheet(title="Dataset_Node_Summary")
    ws_sum.append(list(df_node_summary.columns))
    for row in df_node_summary.itertuples(index=False):
        ws_sum.append(list(row))
    for col_idx, col_name in enumerate(df_node_summary.columns, 1):
        cell = ws_sum.cell(row=1, column=col_idx)
        cell.font = font_header
        cell.alignment = align_center
        cell.border = thin_border
        cell.fill = fill_common
        ws_sum.column_dimensions[get_column_letter(col_idx)].width = 20

wb.save(excel_path)
print(f"✅ エッジ & 孤立ノード統合 Excel 保存完了: {excel_path}")

if not df_isolated.empty:
    wb_iso = openpyxl.Workbook()
    ws_iso_only = wb_iso.active
    ws_iso_only.title = "Isolated_Nodes"
    ws_iso_only.append(list(df_isolated.columns))
    for row in df_isolated.itertuples(index=False):
        ws_iso_only.append(list(row))
    wb_iso.save(iso_excel_path)
    print(f"✅ 孤立ノード専用 Excel 保存完了: {iso_excel_path}")

# ----------------------------------------------------
# 🔄 Kaggle Dataset への進捗自動書き込み同期 (Auto-Sync)
# ----------------------------------------------------
def upload_checkpoint_to_kaggle_dataset(dataset_slug, commit_notes=None):
    """
    Kaggle API 経由で、生成された進捗データ (progress.json, gt_edges_summary.csv 等) を
    Kaggle Dataset (DATASET_SLUG) へ自動アップロード (新バージョン発行) して書き込み同期します。
    """
    is_kaggle = os.path.exists("/kaggle/working")
    if not is_kaggle or not CONTINUOUS_FLAG:
        print("ローカル環境または CONTINUOUS_FLAG=False のため Dataset 自動アップロードをスキップします。")
        return
        
    print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] >>> upload_checkpoint_to_kaggle_dataset 開始")
    from kaggle_secrets import UserSecretsClient
    import kaggle
    
    username = UserSecretsClient().get_secret("KAGGLE_USERNAME")
    key = UserSecretsClient().get_secret("KAGGLE_KEY")
    if not username or not key:
        raise ValueError("❌ [ERROR] Kaggle Secrets に KAGGLE_USERNAME または KAGGLE_KEY が設定されていません。")

    os.environ["KAGGLE_USERNAME"] = username
    os.environ["KAGGLE_KEY"] = key
    
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    
    checkpoint_dir = os.path.join("/kaggle/working", "checkpoint_upload_tmp")
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    try:
        sync_files = ["progress.json", "gt_edges_summary.csv", "gt_isolated_nodes_summary.csv"]
        for sf in sync_files:
            if os.path.exists(sf):
                shutil.copy2(sf, os.path.join(checkpoint_dir, sf))
                print(f"  - Checkpoint コピー対象に追加: {sf}")
                
        meta_path = os.path.join(checkpoint_dir, "dataset-metadata.json")
        meta = {
            "title": dataset_slug,
            "id": f"{username}/{dataset_slug}",
            "licenses": [{"name": "CC0-1.0"}]
        }
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2)
            
        notes = commit_notes if commit_notes else "Auto-update s107 progress checkpoint from Kaggle execution"
        print(f"Uploading updated dataset version to {username}/{dataset_slug}...")
        api.dataset_create_version(checkpoint_dir, version_notes=notes)
        print("✅ Successfully updated Kaggle Dataset checkpoint version!")
    finally:
        if os.path.exists(checkpoint_dir):
            shutil.rmtree(checkpoint_dir, ignore_errors=True)
        
    print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] <<< upload_checkpoint_to_kaggle_dataset 終了")

if CONTINUOUS_FLAG:
    upload_checkpoint_to_kaggle_dataset(DATASET_SLUG)

print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] <<< Cell 7: 最終データ保存 終了")


In [ ]:
# === Cell 8: 🎨 3×3 大幅拡張 EDA & 孤立ノード比較可視化 ===
import datetime
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] >>> Cell 8: 3×3 拡張EDAプロット描画 開始")

if len(df_edges) > 0:
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    fig, axes = plt.subplots(3, 3, figsize=(18, 14), dpi=200)
    fig.suptitle("🧬 Biohub GT Edge & Node Dynamic Feature Analysis (3×3 Grid)", fontsize=18, fontweight='bold', y=0.98)
    
    # --- 1. [1, 1] 3D移動距離分布 (move vs division) ---
    ax = axes[0, 0]
    moves = df_edges[df_edges['edge_type'] == 'move']['track_distance_3d_um']
    divs = df_edges[df_edges['edge_type'] == 'division']['track_distance_3d_um']
    ax.hist(moves, bins=30, alpha=0.7, label=f'Move (N={len(moves)})', color='#1f77b4', edgecolor='black')
    if len(divs) > 0:
        ax.hist(divs, bins=30, alpha=0.7, label=f'Division (N={len(divs)})', color='#ff7f0e', edgecolor='black')
    ax.set_title("(1) 3D Movement Distance (μm)", fontsize=12, fontweight='bold')
    ax.set_xlabel("Distance (μm)")
    ax.set_ylabel("Count")
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    
    # --- 2. [1, 2] 輝度相関 (s_mean_intensity vs e_mean_intensity) ---
    ax = axes[0, 1]
    sc = ax.scatter(df_edges['s_mean_intensity'], df_edges['e_mean_intensity'], c=df_edges['track_distance_3d_um'], cmap='viridis', alpha=0.6, edgecolors='none', s=20)
    lims = [min(df_edges['s_mean_intensity'].min(), df_edges['e_mean_intensity'].min()), max(df_edges['s_mean_intensity'].max(), df_edges['e_mean_intensity'].max())]
    ax.plot(lims, lims, 'r--', alpha=0.75, label='y = x (No Change)')
    ax.set_title("(2) Start vs End Mean Intensity", fontsize=12, fontweight='bold')
    ax.set_xlabel("Start Mean Intensity (s_mean_intensity)")
    ax.set_ylabel("End Mean Intensity (e_mean_intensity)")
    fig.colorbar(sc, ax=ax, label='Distance (μm)')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    
    # --- 3. [1, 3] SNR相関 (s_snr vs e_snr) ---
    ax = axes[0, 2]
    sc = ax.scatter(df_edges['s_snr'], df_edges['e_snr'], c=df_edges['track_distance_3d_um'], cmap='plasma', alpha=0.6, edgecolors='none', s=20)
    lims = [min(df_edges['s_snr'].min(), df_edges['e_snr'].min()), max(df_edges['s_snr'].max(), df_edges['e_snr'].max())]
    ax.plot(lims, lims, 'r--', alpha=0.75, label='y = x')
    ax.set_title("(3) Start vs End SNR", fontsize=12, fontweight='bold')
    ax.set_xlabel("Start SNR (s_snr)")
    ax.set_ylabel("End SNR (e_snr)")
    fig.colorbar(sc, ax=ax, label='Distance (μm)')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    
    # --- 4. [2, 1] Z相対深度相関 (s_z_depth_ratio vs e_z_depth_ratio) ---
    ax = axes[1, 0]
    e_z_depth = df_edges['e_z_depth_ratio'] if 'e_z_depth_ratio' in df_edges.columns else (df_edges['e_z'] / 30.0)
    sc = ax.scatter(df_edges['s_z_depth_ratio'], e_z_depth, c=df_edges['track_distance_3d_um'], cmap='cividis', alpha=0.6, edgecolors='none', s=20)
    ax.plot([0, 1], [0, 1], 'r--', alpha=0.75, label='y = x')
    ax.set_title("(4) Start vs End Z-Depth Ratio", fontsize=12, fontweight='bold')
    ax.set_xlabel("Start Z-Depth Ratio (0=shallow, 1=deep)")
    ax.set_ylabel("End Z-Depth Ratio (0=shallow, 1=deep)")
    fig.colorbar(sc, ax=ax, label='Distance (μm)')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    
    # --- 5. [2, 2] 細胞半径相関 (s_estimated_radius_um vs e_estimated_radius_um) ---
    ax = axes[1, 1]
    sc = ax.scatter(df_edges['s_estimated_radius_um'], df_edges['e_estimated_radius_um'], c=df_edges['track_distance_3d_um'], cmap='magma', alpha=0.6, edgecolors='none', s=20)
    lims = [min(df_edges['s_estimated_radius_um'].min(), df_edges['e_estimated_radius_um'].min()), max(df_edges['s_estimated_radius_um'].max(), df_edges['e_estimated_radius_um'].max())]
    ax.plot(lims, lims, 'r--', alpha=0.75, label='y = x')
    ax.set_title("(5) Start vs End Radius (μm)", fontsize=12, fontweight='bold')
    ax.set_xlabel("Start Radius (μm)")
    ax.set_ylabel("End Radius (μm)")
    fig.colorbar(sc, ax=ax, label='Distance (μm)')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    
    # --- 6. [2, 3] 細胞体積相関 (s_volume_um3 vs e_volume_um3) ---
    ax = axes[1, 2]
    sc = ax.scatter(df_edges['s_volume_um3'], df_edges['e_volume_um3'], c=df_edges['track_distance_3d_um'], cmap='coolwarm', alpha=0.6, edgecolors='none', s=20)
    lims = [min(df_edges['s_volume_um3'].min(), df_edges['e_volume_um3'].min()), max(df_edges['s_volume_um3'].max(), df_edges['e_volume_um3'].max())]
    ax.plot(lims, lims, 'r--', alpha=0.75, label='y = x')
    ax.set_title("(6) Start vs End Volume (μm³)", fontsize=12, fontweight='bold')
    ax.set_xlabel("Start Volume (μm³)")
    ax.set_ylabel("End Volume (μm³)")
    fig.colorbar(sc, ax=ax, label='Distance (μm)')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    
    # --- 7. [3, 1] 局所密度相関 (s_local_density_r15 vs e_local_density_r15) ---
    ax = axes[2, 0]
    s_dens_j = df_edges['s_local_density_r15'] + np.random.uniform(-0.2, 0.2, len(df_edges))
    e_dens_j = df_edges['e_local_density_r15'] + np.random.uniform(-0.2, 0.2, len(df_edges))
    sc = ax.scatter(s_dens_j, e_dens_j, c=df_edges['track_distance_3d_um'], cmap='spring', alpha=0.5, edgecolors='none', s=15)
    max_d = max(df_edges['s_local_density_r15'].max(), df_edges['e_local_density_r15'].max())
    ax.plot([0, max_d], [0, max_d], 'r--', alpha=0.75, label='y = x')
    ax.set_title("(7) Start vs End Local Density (r=15px)", fontsize=12, fontweight='bold')
    ax.set_xlabel("Start Density (count)")
    ax.set_ylabel("End Density (count)")
    fig.colorbar(sc, ax=ax, label='Distance (μm)')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    
    # --- 8. [3, 2] 時間別密度変遷 (Time t vs Mean Density) ---
    ax = axes[2, 1]
    t_density = df_edges.groupby('t')['s_local_density_r15'].agg(['mean', 'std']).reset_index()
    ax.plot(t_density['t'], t_density['mean'], marker='o', color='#2ca02c', linewidth=2, label='Mean Local Density')
    ax.fill_between(t_density['t'], t_density['mean'] - t_density['std'], t_density['mean'] + t_density['std'], color='#2ca02c', alpha=0.2, label='±1 STD')
    ax.set_title("(8) Local Cell Density Transition Over Time (t)", fontsize=12, fontweight='bold')
    ax.set_xlabel("Time Frame (t)")
    ax.set_ylabel("Mean Density (r=15px)")
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    
    # --- 9. [3, 3] 始点SNR vs 3D移動距離 ---
    ax = axes[2, 2]
    sc = ax.scatter(df_edges['s_snr'], df_edges['track_distance_3d_um'], c=df_edges['s_z_depth_ratio'], cmap='twilight', alpha=0.6, edgecolors='none', s=20)
    ax.set_title("(9) Start SNR vs 3D Distance (μm)", fontsize=12, fontweight='bold')
    ax.set_xlabel("Start SNR (s_snr)")
    ax.set_ylabel("3D Distance (μm)")
    fig.colorbar(sc, ax=ax, label='Z-Depth Ratio')
    ax.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig("gt_edges_eda_plot.png", dpi=200)
    plt.show()
    print("✅ 3×3 拡張 EDA プロットを gt_edges_eda_plot.png に保存しました。")
    
    print("\n--- GT Edge 要約統計量 ---")
    print(df_edges[['track_distance_3d_um', 's_snr', 'e_snr', 's_mean_intensity', 'e_mean_intensity', 's_volume_um3', 'e_volume_um3']].describe())

# --- 孤立ノード 比較 EDA プロット ---
if not df_isolated.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=150)
    fig.suptitle("🔍 Connected Nodes vs Isolated Nodes Comparison", fontsize=14, fontweight='bold')
    
    ax = axes[0]
    ax.hist(df_edges['s_snr'], bins=25, alpha=0.6, label='Connected Nodes (Start SNR)', color='blue', density=True)
    ax.hist(df_isolated['snr'], bins=25, alpha=0.6, label='Isolated Nodes (SNR)', color='red', density=True)
    ax.set_title("SNR Density Distribution")
    ax.set_xlabel("SNR")
    ax.set_ylabel("Normalized Density")
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    
    ax = axes[1]
    ax.hist(df_edges['s_local_density_r15'], bins=20, alpha=0.6, label='Connected Nodes', color='blue', density=True)
    ax.hist(df_isolated['local_density_r15'], bins=20, alpha=0.6, label='Isolated Nodes', color='red', density=True)
    ax.set_title("Local Cell Density (r=15px) Distribution")
    ax.set_xlabel("Density (count)")
    ax.set_ylabel("Normalized Density")
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig("gt_isolated_nodes_eda_plot.png", dpi=150)
    plt.show()
    print("✅ 孤立ノード比較 EDA プロットを gt_isolated_nodes_eda_plot.png に保存しました。")

print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] <<< Cell 8: 3×3 拡張EDA 終了")
